In [38]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

## Verify GPU

In [39]:
import torch

print("PyTorch:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
GPU Available: True
GPU: Tesla T4


In [40]:
!pip install transformers datasets accelerate wandb huggingface_hub

## Verify Packages

In [41]:
import transformers
import datasets
import torch

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)

Transformers: 5.0.0
Datasets: 4.8.5
Torch: 2.10.0+cu128


## Load Secrets and Check Login

In [42]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import os
import wandb

secrets = UserSecretsClient()

os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")

wandb.login()

HF_TOKEN = secrets.get_secret("HF_TOKEN")

login(token=HF_TOKEN)

print("Login successful")


Login successful


## Load Dataset

In [43]:
# 1. Load Dataset
from datasets import load_dataset
import json

dataset = load_dataset("stanfordnlp/imdb", token=HF_TOKEN)

# 2. Dataset Inspection
print(dataset)

print("Train size:", len(dataset["train"]))
print("Test size:", len(dataset["test"]))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Train size: 25000
Test size: 25000


## Load Tokenizer

In [44]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded")

Tokenizer loaded


## Tokenize Dataset

In [45]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed")


Tokenization completed


## Remove Text Column

In [46]:
tokenized_dataset = tokenized_dataset.remove_columns(
    ["text"]
)

tokenized_dataset.set_format("torch")

print("Dataset formatted")

Dataset formatted


# Verify Tokenized Dataset

In [47]:
print(tokenized_dataset["train"][0])


{'label': tensor(0), 'input_ids': tensor([  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
         2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
         2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
         2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
         1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
         2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
         6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
         1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
         5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
        14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
         1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
         2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779,
        25430, 14728,  2245,  

In [48]:
print(tokenized_dataset["train"][0].keys())

dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])


# Create Dataset

In [49]:
# train_data = (
#     tokenized_dataset["train"]
#     .shuffle(seed=42)
#     .select(range(10000))
# )

# test_data = (
#     tokenized_dataset["test"]
#     .shuffle(seed=42)
#     .select(range(2000))
# )

train_data = tokenized_dataset["train"]
test_data = tokenized_dataset["test"]

print(len(train_data))
print(len(test_data))

25000
25000


# Load Model

In [50]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

print("Model loaded")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


# Metrics:

In [51]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score

def compute_metrics(pred):

    labels = pred.label_ids

    preds = pred.predictions.argmax(-1)

    return {
        "accuracy":
            accuracy_score(labels, preds),

        "f1":
            f1_score(
                labels,
                preds,
                average="weighted"
            )
    }

# Initialize W&B

In [52]:
import wandb

wandb.init(
    project="mlops-imdb-sentiment",

    name="run-v2",

    config={
        "epochs":3,
        "batch_size":16,
        "learning_rate":5e-5,
        "version":"v2"
    }
)

# Training arguments:

In [53]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_v2",

    num_train_epochs=3,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="wandb",

    run_name="run-v2"
)

## Trainer

In [54]:
from transformers import Trainer

trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_data,

    eval_dataset=test_data,

    compute_metrics=compute_metrics
)


print(tokenized_dataset["train"][0].keys)

print("Trainer created")


<built-in method keys of dict object at 0x7ea5cb628a00>
Trainer created


# Train

In [55]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.627360,0.495736,0.897320,0.897042
2,0.282425,0.584149,0.905840,0.905774
3,0.151340,0.767332,0.913160,0.913156


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2346, training_loss=0.33309169681480777, metrics={'train_runtime': 1276.5166, 'train_samples_per_second': 58.754, 'train_steps_per_second': 1.838, 'total_flos': 4967527449600000.0, 'train_loss': 0.33309169681480777, 'epoch': 3.0})

## Evaluate

In [56]:
results = trainer.evaluate()

print(results)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.4954911470413208, 'eval_accuracy': 0.89748, 'eval_f1': 0.8972041741281878, 'eval_runtime': 103.0063, 'eval_samples_per_second': 242.704, 'eval_steps_per_second': 7.592, 'epoch': 3.0}


## Finish W&B

In [57]:
wandb.finish()

eval/accuracy,▁▅█▁
eval/f1,▁▅█▁
eval/loss,▁▃█▁
eval/runtime,█▆▄▁
eval/samples_per_second,▁▃▅█
eval/steps_per_second,▁▃▅█
train/epoch,▁▂▃▅▅▇███
train/global_step,▁▂▃▅▅▇███
train/grad_norm,██▅▁
train/learning_rate,█▆▃▁
+1,...


## Push Best Model to Hugging Face

In [58]:
# Adding the right labelling
model.config.id2label = {
    0: "NEGATIVE",
    1: "POSITIVE"
}

model.config.label2id = {
    "NEGATIVE": 0,
    "POSITIVE": 1
}

In [59]:
model.push_to_hub(
    "rakeshlrng/imdb-sentiment"
)

tokenizer.push_to_hub(
    "rakeshlrng/imdb-sentiment"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/rakeshlrng/imdb-sentiment/commit/21204220af153048f2b66f5c2df6f3b3880b2fc6', commit_message='Upload tokenizer', commit_description='', oid='21204220af153048f2b66f5c2df6f3b3880b2fc6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/rakeshlrng/imdb-sentiment', endpoint='https://huggingface.co', repo_type='model', repo_id='rakeshlrng/imdb-sentiment'), pr_revision=None, pr_num=None)